# Week 7 Lab Assignment

**What is the ask?**  Implement your customized human motion classification neural network in hardware.

**Why is this important**  In this week's programming assignment and lab assignment you experience the full engineering and data science lifecycle associated with EdgeAI/TinyML design. This experience supports both theoretical understanding and practical application across several core domains, including embedded systems, signal processing, data science, and machine learning.  In short: this week, you tie seven weeks of knowledge development together into a fully functioning system design!    

## Part 1 - STM32CubeMX Project Code Generator

Thus far in the course you have been working at the register level.  While this is common practice in industry, we do not always start there.  Sometimes we just want to get a project off the ground and get a "proof of concept" up and running first.  For this, we can use automatic code generation tools and frameworks.  These are usually provided by the chip vendor, and vary in their sophistication and capabilities. ST Microelectronics, makers of the STM32F042K6 microcontroller we are working with, have a fairly advanced code generation tool: STM32CubeMX.  This, combined with libraries for everything from modifying the system clock to using the USART/SPI interfaces... to incorporating tensorflow-lite models into the design... can greatly accelerate your time to "proof of concept."  

Let's get it installed and take it for a test-drive:

[Install CubeMX](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=1ecdfc02-d6cb-4594-99d4-b3f1015fd960)

You will use a new development board for this lab... thus far you've been working with a NUCLEO-STM32F042K6 development board... the new development board switches from an ARM Cortex-M0 processing core with no hardware floating point, 6kb RAM, and 36kB of flash to an ARM Cortex-M4 processing core *with* hardware floating point, 64kb of RAM, and 256 kB of flash.  Much higher performance, and more room for additional text/data needs.  

Use CubeMX to create a new project for the NUCLEO-STM32L432KC development board:

***NOTE: at the end of this video I instruct you to save the project in a subfolder of the W26-W7-LA folder called "CubeMXProject."  Do not create this subfolder (if you do I will show you in a later video how to correct this).  Instead, save the project in your .../ENGS-62-workspace/W26-W7-LA/ folder (and save yourself from having to fix it in a future video)***

[CubeMX New Nucleo-L432KC Project](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=7eb77f2a-952c-486c-804f-b3f1015fd9f7)

Now that you have a project template, it is time to start customizing it.  First, import your neural network model (the tensorflow-lite model you created in W26-W7-PA):

[Import TFLite](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=3d6cfe76-16af-4b67-b103-b3f1015fd9a6)

Next, set up a basic timer (as the STM32CubeMX environment takes ownership of SysTick) to blink the on-board LED and configure a GPIO digital output that can be used for code timing experiments with the Analog Discovery 3:

[Code Timing GPIO and TIM6](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=f0e12949-34ae-4ce3-9971-b3f1015fd92f)

And last, check the default USART2 settings (automatically set up by the tool) and configure the SPI1 interface + EXTI interrupt for communicating with the accelerometer:

[SPI and EXTI](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=bb95de28-e199-417f-85da-b3f1015fe08e)

Everything all set up?  Time to run code generation!

[CubeMX Code Generation and Tour](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=d4fd1a71-a611-4123-a5a3-b3f1015fe8fc)






## Checkpoint 1 - STM32CubeMX Project Designer

Open the generated project in VS Code, then build the project template (Terminal->Run Build Task - you should find build tasks for an STM32L432KC target).

The project should build without any warnings or errors, with similar text/data/bss code sizes to my results (below).

Replace the following image with the size report from your build:

<!-- ![CP1 Build Size Report](images/CP1_Build_Size.png) -->
![Brandon Checkpoint 1](images/checkpoint1.png)

*Grading Rubric [2 Points]*
* -2: missing image
* -1: for any *significantly* different text, data, or bss value



## Part 2 - Bring-Up

ST's Hardware Abstraction Layer (HAL) provides drivers for most of the hardware's capabilities.  When working "close to the metal", as you did in prior weeks, the challenge is understanding the hardware and writing your low-level code to interface with it.  When working with abstraction layers , the challenge is learning which functions to call to achieve your design objectives.  That just takes time - searching the code, searching online, and experimenting.  It isn't really beneficial to learn the HAL for a single lab, so I'm going to provide you with a worked example (for three motion classes) that you can use for reference.

Start by cloning my W26-W7-LA-Example project into your ENGS-62-workspace folder.  This includes my model.tflite, and a working project that runs classification (for three, not four, motion classes) in response to a key press.  You can build and run this project on your hardware if you like, and you should use it as a reference and source of template code in your project.

[W26-W7-LA-Example GitHub Repo](https://github.com/ENGS-62/W26-W7-LA-Example) (you can clone this directly from the ENGS-62 GitHub organization, you do not need to fork it)

Here is a walkthrough of the W26-W7-LA-Example project running on my hardware:

[W26-W7-LA-Example Walkthrough](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=2b0b257d-d1fe-4bde-96c2-b3f2016f14bc)

Once you are familair with the W26-W7-LA-Example project, switch back to your W26-W7-LA project and start bringing up the software.  First, start by blinking the LED.

### Blink the LED (TIM6)

Bring over the HAL_TIM_PeriodElapsedCallback() function from my W26-W7-LA-Example:

``` C
// Generic HAL timer callback function
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim) {
  // Toggle the LED connected to PB3 every time the TIM6 timer period elapses
  if (htim->Instance == TIM6) {
    HAL_GPIO_TogglePin(GPIOB, GPIO_PIN_3);
  }
}
```

Also bring over the code in main() that starts the timer:

``` C
  // Start the Timer6 timer in interrupt mode
  HAL_TIM_Base_Start_IT(&htim6);
```

Now build and run your code.  

***NOTE: you must change the Target Device type to STM32L432KC in the SEGGER J-Link app when working with the NUCELO-STM32L432KC dev board.  You must change it back to STM32F042K6 when working with the NUCLEO-STM32F0423K6 dev board!***

![J-Link GDB Change Target Device](images/P2_JLink_Target_Device.png)

You should see the on-board LED blinking once per second.  

### Provide __io_putchar() for printf()

'HAL_UART_Transmit()' is ST's HAL function to send one (or more) bytes of data via a USART.  It wants a pointer to the USART instance (e.g. huart1, huart2), a pointer to a buffer of data, the buffer length, and a timeout value as the HAL_UART_Transmit() is capable of "timing out" if the data isn't sent within a specified timeout limit.  

Bring over the __io_putchar() function from W26-W7-LA-Example:

``` C
/* Retarget printf to use USART2 for character output */
int __io_putchar(int data)
{
  // Send one byte at a time, using the maximum timeout delay possible
  HAL_UART_Transmit(&huart2, (uint8_t *)&data, 1, HAL_MAX_DELAY);
  return data;
}
```

Then add the banner printf() statement to main(), build, and run your code.  You should see the banner displayed in your serial terminal emulator when you run your code.

### USART Receive Support

Bring over the HAL_UART_RxCpltCallback() function from W26-W7-LA-Example, as well as the calls to HAL_UART_Receive_IT() in main().  You also need to bring over some of the code in the private variables section of main.c.

Build and run your code.  When you press a key you should see the "Collecting Data for Inference..." message displayed.  

### LIS3DH Accelerometer

Wire up the LIS3DH Accelerometer.  Note that the STM32L432KC uses different IO pins then the STM32042K6 for the SPI1 communications interface!

![Pinouts](images/P2_Pinouts.png)

Now bring over the following from the W26-W7-LA-Example project:

* HAL_GPIO_EXTI_Callback()
* accel_txfr()
* accel_init()
* accel_read()

You also need to bring over some of the private variables and private definitions from the W26-W7-LA-Example project's main.c.

Add the call to accel_init() in main().  Build and run your code.  When you press a key you should see 26 rows of acceleration data displayed to your serial console over a period of about one second.

### AI Inference

Finally bring over the bits and pieces you need for AI inference.  You'll have to modify this code to support your four motion classe, but the parts you need include:

* Private includes related to the AI model
* Private variables used by inference code
* AI_init() and AI_Run() functions
* argmax() function

Update the conditional code block in main() to support your fourth motion class.  Build, test, and debug your code until you have it working with your forth motion class, running a single inference in response to a single keypress.  


## Checkpoint 2 - Working Single-shot Inference

Demonstrate that you have everything working with your project code by capturing a screenshot of the full serial console output, out of system reset, including the response to a single key press.  It should resemble my image, below, but your output should include information about the fourth motion class you created:

<!-- ![Checkpoint 2 Keypress Output](images/CP2_Keypress_Output.png) -->
![Brandon Checkpoint 2 Keypress](images/checkpoint2(CorrectMotion).png)

*Grading Rubric [10 points]*
* -10: missing image
* -5: image does not show raw acceleration output
* -5: inference results do not include results for fourth motion class (different from my example project)

## Part 3 - Continuous Sampling and Inference

Enhance your project to:

* Continuously save all sampled accelerometer data into data buffers (remove all code that waits for a keypress)
* When a sufficient batch of new data (26 samples) is available, run inference on the new data

You should use the "double buffering" approach for this (see Week 5 videos on DMA and Double Buffering, though you are not using DMA in this case - just double buffering).  

The idea is to fill the first data buffer in the EXTI interrupt handler, then run inference on the first data buffer in main() while filling the second data buffer in the EXTI interrupt handler.  When the second data buffer is full, run inference on the second data buffer in main() and switch back to filling the first data buffer in the EXTI interrupt handler.  

There are multiple ways to acheive this.  Be sure to state your design decisions and describe your implementation in your code comments!

### Checkpoint 3 - Continuous Inference

Remove the diagnostic printf() in the EXTI handler that displays raw acceleration data.  Demonstrate that continuous inference is working by replacing my image below with a similar image from your design:

<!-- ![Inference Output](images/CP3_Inference_Output.png) -->
![Brandon Inference Output](images/checkpoint3(TerminalResults).png)

*Grading Rubric [10 points]*
* -10: missing image
* -5: image does not demonstrate continuous inference
* -5: inference results do not include results for fourth motion class (different from my example project)

Your inference values should update about once per second (every 26 samples, sampling at 25 Hz).  This means you have about 1 second to run inference (all code inside of the conditional 'if(data_read_for_inference) {...}' code block, including printf() statements!).  

Toggle the PORT B Pin #5 IO pin as you enter and exit the conditional 'if( data_ready_for_inference) {...}' code block.  Capture the timing information using the WaveForms Oscilloscope utility to demonstrate that you "make timing" within the ~1 second window available for inference:

![Brandon Make Timing](images/checkpoint3(MakeTiming).png) \
*Since the pin goes high to low is about 14 ms it is safe to say that we make timing in the 1 second window available for inference.*

*Grading Rubric [5 points]*
* -5: missing image
* -2: Incorrect trigger configuration
* -2: Timebase and X-axis offset does not maximize captured signal display
* -1: Y-axis does not show 0 and +3.3V voltage levels

If your code is "making timing" (running inference including all printf() output) then congratulations, you've finished Lab 7 and I look forward to seeing your design in action!